In [1]:
#day_1
import uuid
import enum
from abc import ABC, abstractmethod
from decimal import Decimal

# 1. Исключения:
class AccountError(Exception):
    """Базовое исключение для всех ошибок счёта."""
    pass

class AccountFrozenError(AccountError):
    """Операция над замороженным счётом."""
    pass

class AccountClosedError(AccountError):
    """Операция над закрытым счётом."""
    pass

class InvalidOperationError(AccountError):
    """Некорректная операция (сумма, валюта, тип)."""
    pass

class InsufficientFundsError(AccountError):
    """Недостаточно средств на счёте."""
    pass


# 2. Статусы и валюты через Enum
class AccountStatus(enum.Enum):
    ACTIVE = "активный"
    FROZEN = "замороженный"
    CLOSED = "закрытый"

class Currency(enum.Enum):
    RUB = "RUB"
    USD = "USD"
    EUR = "EUR"
    KZT = "KZT"
    CNY = "CNY"

# 3. Абстрактный класс
class AbstractAccount(ABC):
    """Базовый контракт для любого банковского счёта."""

    def __init__(self, owner: str, account_id: str | None = None,
                 currency: Currency = Currency.RUB):
        self.owner = owner
        self.account_id = account_id or self._generate_id()
        self._balance = Decimal("0")
        self._status = AccountStatus.ACTIVE
        self.currency = currency

    # Абстрактные методы
    @abstractmethod
    def deposit(self, amount) -> None:
        ...

    @abstractmethod
    def withdraw(self, amount) -> None:
        ...

    @abstractmethod
    def get_account_info(self) -> str:
        ...

    # Общие утилиты
    @staticmethod
    def _generate_id() -> str:
        """Короткий UUID (8 символов)."""
        return uuid.uuid4().hex[:8].upper()

# 4. Конкретный счёт
class BankAccount(AbstractAccount):
    """Реализация банковского счёта с бизнес-правилами."""

    def __init__(self, owner: str, account_id: str | None = None,
                 currency: Currency = Currency.RUB,
                 initial_balance: float | Decimal = 0):
        # Валидация владельца
        if not owner or not isinstance(owner, str) or not owner.strip():
            raise InvalidOperationError("Имя владельца не может быть пустым")

        # Валидация валюты
        if not isinstance(currency, Currency):
            raise InvalidOperationError(
                f"Валюта должна быть из списка {[c.value for c in Currency]}"
            )

        super().__init__(owner.strip(), account_id, currency)

        # Начальное пополнение, если задано
        if initial_balance:
            self.deposit(initial_balance)

    # Валидация
    @staticmethod
    def _validate_amount(amount) -> Decimal:
        """Проверяем, что сумма - положительное число."""
        if isinstance(amount, bool):  # bool — подкласс int, отсекаем
            raise InvalidOperationError("Сумма должна быть числом, не bool")
        try:
            value = Decimal(str(amount))
        except Exception:
            raise InvalidOperationError(f"Некорректная сумма: {amount!r}")
        if value <= 0:
            raise InvalidOperationError("Сумма должна быть больше нуля")
        return value

    def _ensure_active(self) -> None:
        """Проверяем статус счёта перед операцией."""
        if self._status == AccountStatus.FROZEN:
            raise AccountFrozenError(
                f"Счёт {self.account_id} заморожен, операции запрещены"
            )
        if self._status == AccountStatus.CLOSED:
            raise AccountClosedError(
                f"Счёт {self.account_id} закрыт"
            )

    # Операции
    def deposit(self, amount) -> None:
        self._ensure_active()
        value = self._validate_amount(amount)
        self._balance += value

    def withdraw(self, amount) -> None:
        self._ensure_active()
        value = self._validate_amount(amount)
        if value > self._balance:
            raise InsufficientFundsError(
                f"Недостаточно средств: доступно {self._balance} {self.currency.value}, "
                f"запрошено {value}"
            )
        self._balance -= value

    # Управление статусом
    def freeze(self) -> None:
        if self._status == AccountStatus.CLOSED:
            raise AccountClosedError("Нельзя заморозить закрытый счёт")
        self._status = AccountStatus.FROZEN

    def activate(self) -> None:
        if self._status == AccountStatus.CLOSED:
            raise AccountClosedError("Нельзя активировать закрытый счёт")
        self._status = AccountStatus.ACTIVE

    def close(self) -> None:
        if self._balance != 0:
            raise InvalidOperationError(
                "Нельзя закрыть счёт с ненулевым балансом"
            )
        self._status = AccountStatus.CLOSED

    # Информация
    def get_account_info(self) -> str:
        return (
            f"Владелец: {self.owner}\n"
            f"Счёт: ...{self.account_id[-4:]}\n"
            f"Статус: {self._status.value}\n"
            f"Баланс: {self._balance} {self.currency.value}"
        )

    def __str__(self) -> str:
        return (
            f"[{self.__class__.__name__}] "
            f"{self.owner} | "
            f"...{self.account_id[-4:]} | "
            f"{self._status.value} | "
            f"{self._balance} {self.currency.value}"
        )

    def __repr__(self) -> str:
        return (f"BankAccount(owner={self.owner!r}, "
                f"account_id={self.account_id!r}, "
                f"currency={self.currency.value}, "
                f"balance={self._balance})")


# 5. Демонстрация
def demo() -> None:
    print("=" * 60)
    print("ДЕМОНСТРАЦИЯ РАБОТЫ БАНКОВСКИХ СЧЕТОВ")
    print("=" * 60)

    # 1. Активный счёт
    acc1 = BankAccount("Иван Петров", currency=Currency.RUB,
                       initial_balance=1000)
    print("\n[1] Создан активный счёт:")
    print(acc1)

    # 2. Замороженный счёт
    acc2 = BankAccount("Мария Сидорова", currency=Currency.USD,
                       initial_balance=500)
    acc2.freeze()
    print("\n[2] Создан и заморожен счёт:")
    print(acc2)

    # 3. Операции над замороженным
    print("\n[3] Попытка операций над замороженным счётом:")
    try:
        acc2.deposit(100)
    except AccountFrozenError as e:
        print(f"  ❄️  {e}")
    try:
        acc2.withdraw(50)
    except AccountFrozenError as e:
        print(f"  ❄️  {e}")

    # 4. Валидные операции
    print("\n[4] Валидные операции по активному счёту:")
    acc1.deposit(2500)
    print(f"  После пополнения: {acc1}")
    acc1.withdraw(300)
    print(f"  После снятия:     {acc1}")

    # 5. Ошибки
    print("\n[5] Проверка исключений:")
    for desc, action in [
        ("Отрицательное пополнение", lambda: acc1.deposit(-100)),
        ("Нулевое снятие",           lambda: acc1.withdraw(0)),
        ("Снятие больше баланса",    lambda: acc1.withdraw(999999)),
        ("Некорректная сумма",       lambda: acc1.deposit("abc")),
    ]:
        try:
            action()
            print(f"️ {desc}: ошибка НЕ сработала!")
        except AccountError as e:
            print(f" {desc}: {type(e).__name__} — {e}")

    # 6. Инфо и закрытие
    print("\n[6] Полная информация:")
    print(acc1.get_account_info())

    print("\n[7] Закрытие счёта:")
    acc1.withdraw(acc1._balance)
    acc1.close()
    print(acc1)
    try:
        acc1.deposit(10)
    except AccountClosedError as e:
        print(f"  🚫 {e}")


if __name__ == "__main__":
    demo()

In [2]:
# 1. Нельзя создать абстрактный класс
try:
    AbstractAccount("Тест")
except TypeError as e:
    print("OK:", e)

# 2. Нельзя создать наследника без реализации всех методов
class BadAccount(AbstractAccount):
    def deposit(self, amount): pass
    # забыли withdraw и get_account_info

try:
    BadAccount("Тест")
except TypeError as e:
    print("OK:", e)

✅ OK: Can't instantiate abstract class AbstractAccount with abstract methods deposit, get_account_info, withdraw
✅ OK: Can't instantiate abstract class BadAccount with abstract methods get_account_info, withdraw


In [ ]:
#day_2
